# 04 Experiment 1: Clustering

This notebook evaluates how different text embeddings impact clustering quality. We test the hypothesis that richer, contextual embeddings (BGE, SBERT) yield more coherent clusters than static (GloVe, Word2Vec) or sparse (TF-IDF) ones.

**Pipeline:**
1. Load saved embeddings (`.npy`).
2. Reduce dimensions using UMAP (to 64D for faster clustering).
3. Apply MiniBatchKMeans.
4. Evaluate with NMI, ARI, and Silhouette Score.
5. Visualize with 2D UMAP scatter plots and metric bar charts.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
sys.path.append('..')

from src.cluster import run_clustering_experiment
from src.visualize import plot_umap_2d, plot_metrics_comparison

## 1. Load Data and Ground Truth

In [ ]:
processed_path = '../data/processed/cleaned_reviews.parquet'
df = pd.read_parquet(processed_path)

# Using 'rating' as a proxy for ground truth clusters since we 
# currently only have 'All_Beauty' reviews downloaded. 
# If multiple categories exist, we could use those instead.
labels_true = df['rating'].values
num_clusters = len(np.unique(labels_true))
print(f"Loaded {len(df)} reviews. Found {num_clusters} unique ground truth labels.")

## 2. Run Clustering for All Encoders

In [ ]:
embedding_dir = '../data/embeddings/'
results_dir = '../experiments/clustering/'
figures_dir = '../experiments/figures/'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

encoders = ['tfidf', 'w2v', 'glove', 'sbert', 'bge']
all_metrics = {}
cluster_assignments = {}

# We limit UMAP sampling to 10k points for speed in this notebook.
# In a full 1.5M run, you might increase this slightly, but UMAP is slow.
UMAP_SAMPLE_SIZE = 10000

In [ ]:
for enc in encoders:
    emb_path = os.path.join(embedding_dir, f'{enc}.npy')
    if not os.path.exists(emb_path):
        print(f"Skipping {enc}, embeddings not found at {emb_path}")
        continue
        
    print(f"\n{'='*40}\nEvaluating {enc.upper()}\n{'='*40}")
    X = np.load(emb_path)
    
    # We don't reduce TF-IDF if it's already 300D, but the experiment design says 
    # UMAP to 64D for all. 
    labels_pred, metrics, X_clust = run_clustering_experiment(
        X, 
        labels_true, 
        k=num_clusters, 
        n_components=64, 
        umap_sample_size=UMAP_SAMPLE_SIZE
    )
    
    all_metrics[enc.upper()] = metrics
    cluster_assignments[enc.upper()] = labels_pred
    
    # Optionally save individual metrics
    with open(os.path.join(results_dir, f'{enc}_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)
        
    # Plot 2D UMAP colored by ground truth
    plot_umap_2d(
        X, 
        labels=labels_true, 
        title=f'{enc.upper()} Embeddings (Ground Truth)',
        filename=os.path.join(figures_dir, f'{enc}_umap_gt.png'),
        sample_size=2000 # keep 2D plots fast
    )
    
    # Plot 2D UMAP colored by cluster assignment
    plot_umap_2d(
        X_clust, 
        labels=labels_pred, 
        title=f'{enc.upper()} Clusters (k={num_clusters})',
        filename=os.path.join(figures_dir, f'{enc}_umap_clusters.png'),
        sample_size=2000
    )

## 3. Compare Encoders

In [ ]:
if all_metrics:
    print("\nFinal Metric Summary:")
    for enc, m in all_metrics.items():
        print(f"{enc:10s} - NMI: {m['nmi']:.4f}, ARI: {m['ari']:.4f}, Sil: {m['silhouette']:.4f}")
        
    # Save aggregated metrics
    with open(os.path.join(results_dir, 'all_metrics.json'), 'w') as f:
        json.dump(all_metrics, f, indent=4)

    # Plot Bar Chart
    plot_metrics_comparison(
        all_metrics, 
        filename=os.path.join(figures_dir, 'clustering_metrics_comparison.png')
    )
else:
    print("No encoders evaluated. Run the encode notebook first!")